# Experiment 16: Cross-Condition Calibration Transfer for Embedding-Level Drift Detection

## 1. Objective

This experiment investigates whether embedding-level drift detection thresholds calibrated on one data distribution remain reliable when applied to a different target distribution.

The study compares Euclidean distance and Ledoit–Wolf-regularized Mahalanobis distance under isotropic, anisotropic, and correlated synthetic embedding distributions.

## 2. Research Questions

1. How does transferring a frozen calibration threshold across distribution conditions affect the false-alarm rate?
2. Does calibration transfer affect drift detection sensitivity?
3. Do Euclidean and Mahalanobis detectors differ in their robustness to cross-condition calibration transfer?
4. How do the results compare with condition-specific calibration thresholds?

## 3. Experimental Methodology

The experiment uses synthetic embeddings with a dimensionality of 64 and a window size of 20 embeddings. The evaluation is designed around 20 random seeds, with separate reference, calibration, control, and evaluation data.

The principal stages are:

- Generate synthetic embedding distributions with different covariance structures.
- Calibrate detectors on a source condition and freeze the resulting thresholds.
- Apply the frozen thresholds to independent control and drift-evaluation windows from target conditions.
- Compare transferred-threshold performance against a condition-specific calibration benchmark.
- Aggregate false-alarm rates and detection rates across random seeds.
- Conduct statistical comparisons and numerical integrity checks.

## 4. Detectors

**Euclidean detector:** Measures the Euclidean distance between a window's mean embedding and the reference mean.

**Mahalanobis detector:** Measures the distance using a regularized covariance estimate, accounting for variation and correlation among embedding dimensions.

## 5. Evaluation Metrics

- False-alarm rate on independent control windows.
- Drift detection rate on evaluation windows with injected distribution shifts.
- Difference in detection rates between detectors and calibration strategies.
- Across-seed variability and confidence intervals.
- Numerical stability and integrity audit results.

## 6. Reproducibility

The experiment records configuration parameters, software versions, random seeds, calibration thresholds, evaluation outputs, and validation checks. Results are saved under `results/experiment_16/`.

## 7. Scope and Limitations

This experiment uses synthetic embedding distributions. It evaluates statistical properties of the detectors and their calibration transfer; it does not establish real-world biometric recognition accuracy, fairness, or deployment readiness.

All findings will be reported from the actual experiment outputs. No results are assumed in advance.

In [3]:

from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import scipy
import sklearn

CONFIG = {
    "experiment_id": 16,
    "title": "Cross-Condition Calibration Transfer for Embedding-Level Drift Detection",
    "random_seed": 202616,
    "n_seeds": 20,
    "embedding_dimension": 64,
    "window_size": 20,
    "reference_samples": 1000,
    "calibration_windows": 200,
    "control_windows": 500,
    "evaluation_windows_per_condition": 500,
    "target_false_alarm_rate": 0.05,
    "mean_shift_strength": 0.5,
    "correlation_strength": 0.6,
    "detectors": ["euclidean", "mahalanobis"],
    "conditions": ["isotropic", "anisotropic", "correlated"],
    "calibration_strategy": "source_condition_threshold_frozen"
}

RESULTS_DIR = Path("/content/results/experiment_16")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

environment = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "platform": platform.platform()
}

with open(RESULTS_DIR / "environment.json", "w") as f:
    json.dump(environment, f, indent=2)

print("Experiment 16 configuration created.")
print("Results directory:", RESULTS_DIR)
print("Seeds:", CONFIG["n_seeds"])
print("Embedding dimension:", CONFIG["embedding_dimension"])
print("Conditions:", CONFIG["conditions"])
print("Detectors:", CONFIG["detectors"])
print("Target false-alarm rate:", CONFIG["target_false_alarm_rate"])
print("Environment:", environment)


Experiment 16 configuration created.
Results directory: /content/results/experiment_16
Seeds: 20
Embedding dimension: 64
Conditions: ['isotropic', 'anisotropic', 'correlated']
Detectors: ['euclidean', 'mahalanobis']
Target false-alarm rate: 0.05
Environment: {'python': '3.13.18', 'numpy': '2.4.6', 'pandas': '2.2.3', 'scipy': '1.16.3', 'scikit_learn': '1.6.1', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39'}


## Step 2: Synthetic Covariance Construction and Numerical Validation

This stage constructs three synthetic embedding distributions: isotropic, anisotropic, and correlated.

- **Isotropic:** All embedding dimensions have equal variance.
- **Anisotropic:** Embedding dimensions have different variances.
- **Correlated:** Embedding dimensions exhibit structured pairwise correlations.

The covariance matrices are checked for symmetry, finite values, and positive definiteness. These checks help ensure that the distributions are numerically valid for subsequent detector calibration and evaluation.

No detection results are generated in this stage.

In [4]:

import numpy as np
import pandas as pd

D = CONFIG["embedding_dimension"]
rho = CONFIG["correlation_strength"]

# 1. Isotropic covariance
cov_isotropic = np.eye(D)

# 2. Anisotropic covariance: variances range from 0.25 to 4.0
variances = np.geomspace(0.25, 4.0, D)
cov_anisotropic = np.diag(variances)

# 3. Correlated covariance: Toeplitz structure
indices = np.arange(D)
cov_correlated = rho ** np.abs(indices[:, None] - indices[None, :])

COVARIANCE_MATRICES = {
    "isotropic": cov_isotropic,
    "anisotropic": cov_anisotropic,
    "correlated": cov_correlated,
}

diagnostics = []

for condition, cov in COVARIANCE_MATRICES.items():
    eigenvalues = np.linalg.eigvalsh(cov)
    diagnostics.append({
        "condition": condition,
        "dimension": cov.shape[0],
        "all_finite": bool(np.isfinite(cov).all()),
        "symmetric": bool(np.allclose(cov, cov.T)),
        "minimum_eigenvalue": float(eigenvalues.min()),
        "maximum_eigenvalue": float(eigenvalues.max()),
        "positive_definite": bool(eigenvalues.min() > 1e-10),
        "condition_number": float(np.linalg.cond(cov)),
    })

covariance_diagnostics = pd.DataFrame(diagnostics)
display(covariance_diagnostics)

assert covariance_diagnostics["all_finite"].all()
assert covariance_diagnostics["symmetric"].all()
assert covariance_diagnostics["positive_definite"].all()

covariance_diagnostics.to_csv(
    RESULTS_DIR / "covariance_diagnostics.csv",
    index=False
)

print("Covariance validation passed.")
print("Saved:", RESULTS_DIR / "covariance_diagnostics.csv")


,condition,dimension,all_finite,symmetric,minimum_eigenvalue,maximum_eigenvalue,positive_definite,condition_number
0,isotropic,64,True,True,1.00000,1.000000,True,1.000000
1,anisotropic,64,True,True,0.25000,4.000000,True,16.000000
2,correlated,64,True,True,0.25014,3.968234,True,15.864041


Covariance validation passed.
Saved: /content/results/experiment_16/covariance_diagnostics.csv


## Step 3: Reference Dataset Generation

This stage generates independent reference embeddings for each covariance condition and random seed.

Each reference dataset contains 1,000 samples with 64 dimensions and is drawn from a zero-mean multivariate normal distribution using the corresponding covariance matrix.

The reference datasets establish the baseline distribution used by the drift detectors. Separate random seeds produce independent datasets to support multi-seed evaluation.

This stage records dataset diagnostics only; it does not calibrate thresholds or evaluate drift detection performance.

In [5]:

# Experiment 16 — Step 3: Generate reference datasets

REFERENCE_DATASETS = {}
reference_diagnostics = []

n_reference = CONFIG["reference_samples"]
D = CONFIG["embedding_dimension"]

for seed in range(CONFIG["random_seed"], CONFIG["random_seed"] + CONFIG["n_seeds"]):
    for condition, covariance in COVARIANCE_MATRICES.items():
        rng = np.random.default_rng(seed * 100 + CONFIG["conditions"].index(condition))

        X_ref = rng.multivariate_normal(
            mean=np.zeros(D),
            cov=covariance,
            size=n_reference
        )

        REFERENCE_DATASETS[(seed, condition)] = X_ref

        reference_diagnostics.append({
            "seed": seed,
            "condition": condition,
            "n_samples": X_ref.shape[0],
            "n_dimensions": X_ref.shape[1],
            "all_finite": bool(np.isfinite(X_ref).all()),
            "mean_abs": float(np.abs(X_ref.mean(axis=0)).mean()),
            "average_sample_variance": float(X_ref.var(axis=0).mean()),
        })

reference_diagnostics_df = pd.DataFrame(reference_diagnostics)

reference_diagnostics_df.to_csv(
    RESULTS_DIR / "reference_dataset_diagnostics.csv",
    index=False
)

print("Reference datasets generated:", len(REFERENCE_DATASETS))
print("Expected datasets:", CONFIG["n_seeds"] * len(CONFIG["conditions"]))
print("Dataset shape:", (n_reference, D))
print("All values finite:", all(
    np.isfinite(X).all() for X in REFERENCE_DATASETS.values()
))
print("\nDiagnostics by condition:")
display(
    reference_diagnostics_df.groupby("condition")[
        ["n_samples", "mean_abs", "average_sample_variance"]
    ].mean().round(4)
)

assert len(REFERENCE_DATASETS) == CONFIG["n_seeds"] * len(CONFIG["conditions"])
assert all(X.shape == (n_reference, D) for X in REFERENCE_DATASETS.values())
assert all(np.isfinite(X).all() for X in REFERENCE_DATASETS.values())

print("Reference dataset validation passed.")
print("Saved:", RESULTS_DIR / "reference_dataset_diagnostics.csv")


Reference datasets generated: 60
Expected datasets: 60
Dataset shape: (1000, 64)
All values finite: True

Diagnostics by condition:


,n_samples,mean_abs,average_sample_variance
condition,,,
anisotropic,1000.0,0.0275,1.3629
correlated,1000.0,0.0244,0.9971
isotropic,1000.0,0.0246,0.9990


Reference dataset validation passed.
Saved: /content/results/experiment_16/reference_dataset_diagnostics.csv


## Step 4: Cross-Condition Calibration-Transfer Design

This stage defines the source and target distribution pairs used to evaluate calibration transfer.

The three source conditions are isotropic, anisotropic, and correlated. Each detector will be calibrated on a source condition, and its threshold will then be frozen before evaluation on independent target-condition data.

All nine source-target pairs will be evaluated, including the three same-condition pairs as reference cases and the six cross-condition pairs as transfer cases.

For each pair, the experiment will distinguish between:

- Independent target-condition control windows without injected drift.
- Target-condition evaluation windows with controlled mean shifts.

A condition-specific calibration benchmark will also be included to help distinguish threshold-transfer effects from inherent differences between detectors.

The source and target reference datasets remain separate by condition and seed. Thresholds will not be adjusted using target evaluation outcomes.

In [6]:

# Experiment 16 — Step 4: Define source-target transfer scenarios

CONDITIONS = CONFIG["conditions"]

TRANSFER_SCENARIOS = [
    {
        "source_condition": source,
        "target_condition": target,
        "scenario_type": (
            "same_condition"
            if source == target
            else "cross_condition"
        ),
    }
    for source in CONDITIONS
    for target in CONDITIONS
]

transfer_scenarios_df = pd.DataFrame(TRANSFER_SCENARIOS)

transfer_scenarios_df.to_csv(
    RESULTS_DIR / "calibration_transfer_scenarios.csv",
    index=False
)

print("Source-target scenarios:", len(transfer_scenarios_df))
print("Same-condition scenarios:",
      (transfer_scenarios_df["scenario_type"] == "same_condition").sum())
print("Cross-condition scenarios:",
      (transfer_scenarios_df["scenario_type"] == "cross_condition").sum())

display(transfer_scenarios_df)

assert len(transfer_scenarios_df) == 9
assert (transfer_scenarios_df["scenario_type"] == "same_condition").sum() == 3
assert (transfer_scenarios_df["scenario_type"] == "cross_condition").sum() == 6

print("Calibration-transfer scenario validation passed.")
print("Saved:", RESULTS_DIR / "calibration_transfer_scenarios.csv")


Source-target scenarios: 9
Same-condition scenarios: 3
Cross-condition scenarios: 6


,source_condition,target_condition,scenario_type
0,isotropic,isotropic,same_condition
1,isotropic,anisotropic,cross_condition
2,isotropic,correlated,cross_condition
3,anisotropic,isotropic,cross_condition
4,anisotropic,anisotropic,same_condition
5,anisotropic,correlated,cross_condition
6,correlated,isotropic,cross_condition
7,correlated,anisotropic,cross_condition
8,correlated,correlated,same_condition


Calibration-transfer scenario validation passed.
Saved: /content/results/experiment_16/calibration_transfer_scenarios.csv


## Step 5: Detector Scoring and Frozen Threshold Calibration

This stage implements two embedding-level drift detectors: Euclidean distance and Ledoit–Wolf-regularized Mahalanobis distance.

Each incoming window is represented by its mean embedding. The detector calculates a score measuring its deviation from the reference distribution.

For each random seed, source condition, and detector, the calibration scores are calculated using independent windows generated from the source distribution. A threshold is then selected using the empirical 95th percentile, corresponding to the target false-alarm rate of 5%.

The resulting thresholds are frozen before evaluation on target-condition control and drift windows. Target evaluation data will not be used to tune transferred thresholds.

Calibration scores, threshold values, and supporting metadata are saved for reproducibility.

In [7]:

# Experiment 16 — Step 5: Detector scoring and source calibration

from sklearn.covariance import LedoitWolf

def make_window_means(X, window_size, n_windows, rng):
    """Generate independent windows and return each window's mean embedding."""
    windows = rng.multivariate_normal(
        mean=np.zeros(X.shape[1]),
        cov=np.cov(X, rowvar=False),
        size=(n_windows, window_size)
    )
    return windows.mean(axis=1)


def generate_windows(mean, covariance, n_windows, window_size, rng):
    """Generate window-level embeddings from a specified distribution."""
    samples = rng.multivariate_normal(
        mean=mean,
        cov=covariance,
        size=(n_windows, window_size)
    )
    return samples.mean(axis=1)


def fit_detector(reference):
    """Fit reference mean and regularized covariance."""
    reference_mean = reference.mean(axis=0)
    lw = LedoitWolf().fit(reference)
    precision = lw.precision_
    return reference_mean, precision


def detector_scores(window_means, reference_mean, precision, detector):
    """Calculate Euclidean or Mahalanobis scores."""
    delta = window_means - reference_mean

    if detector == "euclidean":
        return np.linalg.norm(delta, axis=1)

    if detector == "mahalanobis":
        squared = np.einsum("ni,ij,nj->n", delta, precision, delta)
        return np.sqrt(np.maximum(squared, 0.0))

    raise ValueError(f"Unknown detector: {detector}")


CALIBRATION_ROWS = []
THRESHOLD_ROWS = []
FITTED_DETECTORS = {}

n_calibration = CONFIG["calibration_windows"]
window_size = CONFIG["window_size"]
target_fpr = CONFIG["target_false_alarm_rate"]

for seed in range(
    CONFIG["random_seed"],
    CONFIG["random_seed"] + CONFIG["n_seeds"]
):
    for source in CONDITIONS:
        reference = REFERENCE_DATASETS[(seed, source)]
        ref_mean, precision = fit_detector(reference)
        FITTED_DETECTORS[(seed, source)] = (ref_mean, precision)

        rng = np.random.default_rng(seed * 10000 + CONDITIONS.index(source))

        calibration_means = generate_windows(
            mean=np.zeros(D),
            covariance=COVARIANCE_MATRICES[source],
            n_windows=n_calibration,
            window_size=window_size,
            rng=rng
        )

        for detector in CONFIG["detectors"]:
            scores = detector_scores(
                calibration_means, ref_mean, precision, detector
            )

            threshold = float(np.quantile(scores, 1 - target_fpr))

            for window_index, score in enumerate(scores):
                CALIBRATION_ROWS.append({
                    "seed": seed,
                    "source_condition": source,
                    "detector": detector,
                    "window_index": window_index,
                    "score": float(score)
                })

            THRESHOLD_ROWS.append({
                "seed": seed,
                "source_condition": source,
                "detector": detector,
                "target_false_alarm_rate": target_fpr,
                "threshold": threshold,
                "calibration_windows": n_calibration
            })

calibration_df = pd.DataFrame(CALIBRATION_ROWS)
thresholds_df = pd.DataFrame(THRESHOLD_ROWS)

calibration_df.to_csv(
    RESULTS_DIR / "calibration_scores.csv", index=False
)
thresholds_df.to_csv(
    RESULTS_DIR / "frozen_thresholds.csv", index=False
)

print("Calibration score rows:", len(calibration_df))
print("Frozen thresholds:", len(thresholds_df))
print("Expected thresholds:", CONFIG["n_seeds"] * len(CONDITIONS) * len(CONFIG["detectors"]))
print("All calibration scores finite:", np.isfinite(calibration_df["score"]).all())
print("All thresholds finite:", np.isfinite(thresholds_df["threshold"]).all())
display(
    thresholds_df.groupby(["source_condition", "detector"])["threshold"]
    .agg(["mean", "std"])
    .round(4)
)

assert len(thresholds_df) == CONFIG["n_seeds"] * len(CONDITIONS) * len(CONFIG["detectors"])
assert np.isfinite(calibration_df["score"]).all()
assert np.isfinite(thresholds_df["threshold"]).all()

print("Detector calibration validation passed.")
print("Saved calibration scores and frozen thresholds.")


Calibration score rows: 24000
Frozen thresholds: 120
Expected thresholds: 120
All calibration scores finite: True
All thresholds finite: True


mean     std
source_condition detector                   
anisotropic      euclidean    2.4881  0.0331
                 mahalanobis  2.0532  0.0184
correlated       euclidean    2.1902  0.0425
                 mahalanobis  2.0688  0.0313
isotropic        euclidean    2.0565  0.0222
                 mahalanobis  2.0580  0.0237

Detector calibration validation passed.
Saved calibration scores and frozen thresholds.


## Step 6: Independent Control Evaluation Under Calibration Transfer

This stage evaluates the frozen source-condition thresholds on independent target-condition control windows without injected drift.

For each random seed, source condition, target condition, and detector, the experiment calculates the proportion of target control windows whose scores exceed the frozen source threshold.

The same-condition scenarios provide a baseline, while cross-condition scenarios reveal whether transferring a threshold changes the false-alarm rate.

The evaluation uses 500 independent control windows per target condition and does not adjust thresholds based on control outcomes. Results are reported as observed false-alarm rates and compared with the nominal target rate of 5%.

Because the control windows are generated synthetically, these findings measure statistical behavior under the specified simulation assumptions rather than real-world biometric performance.

In [8]:

# Experiment 16 — Step 6: Independent control evaluation

CONTROL_ROWS = []
CONTROL_SUMMARY_ROWS = []

n_control = CONFIG["control_windows"]

threshold_lookup = {
    (int(row.seed), row.source_condition, row.detector): float(row.threshold)
    for row in thresholds_df.itertuples(index=False)
}

for seed in range(
    CONFIG["random_seed"],
    CONFIG["random_seed"] + CONFIG["n_seeds"]
):
    for scenario in TRANSFER_SCENARIOS:
        source = scenario["source_condition"]
        target = scenario["target_condition"]

        ref_mean, precision = FITTED_DETECTORS[(seed, source)]

        # Separate reproducible RNG stream for each source-target pair.
        source_idx = CONDITIONS.index(source)
        target_idx = CONDITIONS.index(target)
        rng = np.random.default_rng(
            seed * 100000 + source_idx * 1000 + target_idx + 50000
        )

        control_means = generate_windows(
            mean=np.zeros(D),
            covariance=COVARIANCE_MATRICES[target],
            n_windows=n_control,
            window_size=window_size,
            rng=rng
        )

        for detector in CONFIG["detectors"]:
            threshold = threshold_lookup[(seed, source, detector)]
            scores = detector_scores(
                control_means, ref_mean, precision, detector
            )
            alarms = scores > threshold
            observed_fpr = float(alarms.mean())

            CONTROL_ROWS.append({
                "seed": seed,
                "source_condition": source,
                "target_condition": target,
                "scenario_type": scenario["scenario_type"],
                "detector": detector,
                "window_index": np.arange(n_control),
                "score": scores,
                "threshold": threshold,
                "alarm": alarms
            })

            CONTROL_SUMMARY_ROWS.append({
                "seed": seed,
                "source_condition": source,
                "target_condition": target,
                "scenario_type": scenario["scenario_type"],
                "detector": detector,
                "n_control_windows": n_control,
                "false_alarms": int(alarms.sum()),
                "observed_false_alarm_rate": observed_fpr,
                "target_false_alarm_rate": target_fpr
            })

control_results_df = pd.DataFrame(CONTROL_ROWS)
control_seed_summary_df = pd.DataFrame(CONTROL_SUMMARY_ROWS)

# Expand arrays stored in the rows into one row per window.
control_results_df = control_results_df.explode(
    ["window_index", "score", "alarm"], ignore_index=True
)

control_results_df["window_index"] = control_results_df["window_index"].astype(int)
control_results_df["score"] = control_results_df["score"].astype(float)
control_results_df["threshold"] = control_results_df["threshold"].astype(float)
control_results_df["alarm"] = control_results_df["alarm"].astype(bool)

control_summary_df = (
    control_seed_summary_df
    .groupby(
        ["source_condition", "target_condition", "scenario_type", "detector"],
        as_index=False
    )
    .agg(
        mean_false_alarm_rate=("observed_false_alarm_rate", "mean"),
        sd_false_alarm_rate=("observed_false_alarm_rate", "std"),
        mean_false_alarms=("false_alarms", "mean"),
        total_control_windows=("n_control_windows", "sum")
    )
)

control_results_df.to_csv(
    RESULTS_DIR / "independent_control_results.csv", index=False
)
control_seed_summary_df.to_csv(
    RESULTS_DIR / "independent_control_seed_summary.csv", index=False
)
control_summary_df.to_csv(
    RESULTS_DIR / "independent_control_summary.csv", index=False
)

expected_windows = (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS)
    * len(CONFIG["detectors"]) * n_control
)

print("Independent control windows:", len(control_results_df))
print("Expected control windows:", expected_windows)
print("Seed-level control summaries:", len(control_seed_summary_df))
print("All scores finite:", np.isfinite(control_results_df["score"]).all())
print("All thresholds finite:", np.isfinite(control_results_df["threshold"]).all())
print("\nMean false-alarm rates by transfer scenario:")
display(
    control_summary_df.pivot_table(
        index=["source_condition", "target_condition", "scenario_type"],
        columns="detector",
        values="mean_false_alarm_rate"
    ).round(4)
)

assert len(control_results_df) == expected_windows
assert len(control_seed_summary_df) == (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS) * len(CONFIG["detectors"])
)
assert np.isfinite(control_results_df["score"]).all()
assert np.isfinite(control_results_df["threshold"]).all()

print("Independent control evaluation validation passed.")
print("Saved independent control results and summaries.")


Independent control windows: 180000
Expected control windows: 180000
Seed-level control summaries: 360
All scores finite: True
All thresholds finite: True

Mean false-alarm rates by transfer scenario:


detector                                           euclidean  mahalanobis
source_condition target_condition scenario_type                          
anisotropic      anisotropic      same_condition      0.0507       0.0516
                 correlated       cross_condition     0.0028       0.3538
                 isotropic        cross_condition     0.0002       0.3408
correlated       anisotropic      cross_condition     0.3284       0.9950
                 correlated       same_condition      0.0488       0.0530
                 isotropic        cross_condition     0.0084       0.9667
isotropic        anisotropic      cross_condition     0.5415       0.5410
                 correlated       cross_condition     0.1210       0.1209
                 isotropic        same_condition      0.0575       0.0573

Independent control evaluation validation passed.
Saved independent control results and summaries.


## Step 7: Drift Detection Under Cross-Condition Calibration Transfer

This stage evaluates whether detectors using frozen source-condition thresholds can detect controlled mean shifts in independent target-condition windows.

For each random seed, source condition, target condition, and detector, evaluation windows are generated with a mean shift applied to the target distribution. The source-condition threshold remains frozen throughout evaluation.

The primary metric is the detection rate: the proportion of drift evaluation windows whose detector scores exceed the frozen threshold.

The results will be compared across same-condition and cross-condition scenarios. High detection rates must be interpreted alongside the independent control false-alarm rates, because a detector that alarms on most control windows may also report high detection rates without providing useful discrimination.

This is a synthetic evaluation and does not establish real-world biometric detection performance.

In [9]:

# Experiment 16 — Step 7: Drift evaluation using frozen source thresholds

DRIFT_ROWS = []
DRIFT_SEED_SUMMARY_ROWS = []

n_eval = CONFIG["evaluation_windows_per_condition"]
shift_strength = CONFIG["mean_shift_strength"]

for seed in range(
    CONFIG["random_seed"],
    CONFIG["random_seed"] + CONFIG["n_seeds"]
):
    for scenario in TRANSFER_SCENARIOS:
        source = scenario["source_condition"]
        target = scenario["target_condition"]

        ref_mean, precision = FITTED_DETECTORS[(seed, source)]
        target_cov = COVARIANCE_MATRICES[target]

        # Use a reproducible target-condition drift direction.
        direction_rng = np.random.default_rng(
            seed * 200000
            + CONDITIONS.index(source) * 1000
            + CONDITIONS.index(target)
            + 70000
        )
        direction = direction_rng.normal(size=D)
        direction /= np.linalg.norm(direction)

        shift = shift_strength * direction

        rng = np.random.default_rng(
            seed * 300000
            + CONDITIONS.index(source) * 1000
            + CONDITIONS.index(target)
            + 90000
        )

        drift_means = generate_windows(
            mean=shift,
            covariance=target_cov,
            n_windows=n_eval,
            window_size=window_size,
            rng=rng
        )

        for detector in CONFIG["detectors"]:
            threshold = threshold_lookup[(seed, source, detector)]
            scores = detector_scores(
                drift_means, ref_mean, precision, detector
            )
            alarms = scores > threshold
            detection_rate = float(alarms.mean())

            DRIFT_ROWS.append({
                "seed": seed,
                "source_condition": source,
                "target_condition": target,
                "scenario_type": scenario["scenario_type"],
                "detector": detector,
                "window_index": np.arange(n_eval),
                "score": scores,
                "threshold": threshold,
                "alarm": alarms
            })

            DRIFT_SEED_SUMMARY_ROWS.append({
                "seed": seed,
                "source_condition": source,
                "target_condition": target,
                "scenario_type": scenario["scenario_type"],
                "detector": detector,
                "n_evaluation_windows": n_eval,
                "detected_windows": int(alarms.sum()),
                "detection_rate": detection_rate
            })

drift_results_df = pd.DataFrame(DRIFT_ROWS).explode(
    ["window_index", "score", "alarm"], ignore_index=True
)
drift_results_df["window_index"] = drift_results_df["window_index"].astype(int)
drift_results_df["score"] = drift_results_df["score"].astype(float)
drift_results_df["threshold"] = drift_results_df["threshold"].astype(float)
drift_results_df["alarm"] = drift_results_df["alarm"].astype(bool)

drift_seed_summary_df = pd.DataFrame(DRIFT_SEED_SUMMARY_ROWS)

drift_summary_df = (
    drift_seed_summary_df
    .groupby(
        ["source_condition", "target_condition", "scenario_type", "detector"],
        as_index=False
    )
    .agg(
        mean_detection_rate=("detection_rate", "mean"),
        sd_detection_rate=("detection_rate", "std"),
        mean_detected_windows=("detected_windows", "mean"),
        total_evaluation_windows=("n_evaluation_windows", "sum")
    )
)

drift_results_df.to_csv(
    RESULTS_DIR / "drift_evaluation_results.csv", index=False
)
drift_seed_summary_df.to_csv(
    RESULTS_DIR / "drift_evaluation_seed_summary.csv", index=False
)
drift_summary_df.to_csv(
    RESULTS_DIR / "drift_evaluation_summary.csv", index=False
)

expected_eval_windows = (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS)
    * len(CONFIG["detectors"]) * n_eval
)

print("Drift evaluation windows:", len(drift_results_df))
print("Expected evaluation windows:", expected_eval_windows)
print("Seed-level drift summaries:", len(drift_seed_summary_df))
print("All scores finite:", np.isfinite(drift_results_df["score"]).all())
print("\nMean detection rates by transfer scenario:")
display(
    drift_summary_df.pivot_table(
        index=["source_condition", "target_condition", "scenario_type"],
        columns="detector",
        values="mean_detection_rate"
    ).round(4)
)

assert len(drift_results_df) == expected_eval_windows
assert len(drift_seed_summary_df) == (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS) * len(CONFIG["detectors"])
)
assert np.isfinite(drift_results_df["score"]).all()
assert np.isfinite(drift_results_df["threshold"]).all()

print("Drift evaluation validation passed.")
print("Saved drift evaluation results and summaries.")


Drift evaluation windows: 180000
Expected evaluation windows: 180000
Seed-level drift summaries: 360
All scores finite: True

Mean detection rates by transfer scenario:


detector                                           euclidean  mahalanobis
source_condition target_condition scenario_type                          
anisotropic      anisotropic      same_condition      0.0877       0.1381
                 correlated       cross_condition     0.0060       0.4504
                 isotropic        cross_condition     0.0002       0.4846
correlated       anisotropic      cross_condition     0.4176       0.9984
                 correlated       same_condition      0.0745       0.2194
                 isotropic        cross_condition     0.0295       0.9846
isotropic        anisotropic      cross_condition     0.6427       0.6425
                 correlated       cross_condition     0.1918       0.1917
                 isotropic        same_condition      0.1254       0.1250

Drift evaluation validation passed.
Saved drift evaluation results and summaries.


## Step 8: Statistical Analysis of Calibration Transfer

This stage compares the Euclidean and Mahalanobis detectors across random seeds and source-target calibration scenarios.

The analysis calculates the difference in detection rates between the two detectors for each seed and scenario, using the paired difference:

Mahalanobis detection rate − Euclidean detection rate.

Mean paired differences and 95% confidence intervals are calculated across the 20 seeds. The analysis also reports the corresponding false-alarm rates so that any increase in detection sensitivity can be interpreted alongside the risk of false alarms.

Confidence intervals are unadjusted for multiple comparisons and should be interpreted accordingly. The results measure performance under the synthetic data-generating assumptions used in this experiment.

In [10]:

# Experiment 16 — Step 8: Paired statistical comparison

from scipy import stats

# Paired detection-rate differences for each seed and scenario
detection_pivot = drift_seed_summary_df.pivot(
    index=["seed", "source_condition", "target_condition", "scenario_type"],
    columns="detector",
    values="detection_rate"
).reset_index()

detection_pivot["mahalanobis_minus_euclidean"] = (
    detection_pivot["mahalanobis"] - detection_pivot["euclidean"]
)

difference_rows = []

group_cols = [
    "source_condition",
    "target_condition",
    "scenario_type"
]

for keys, group in detection_pivot.groupby(group_cols):
    diffs = group["mahalanobis_minus_euclidean"].to_numpy()
    n = len(diffs)
    mean_diff = float(np.mean(diffs))
    sd_diff = float(np.std(diffs, ddof=1))
    se_diff = sd_diff / np.sqrt(n)
    critical_t = stats.t.ppf(0.975, df=n - 1)
    ci_low = mean_diff - critical_t * se_diff
    ci_high = mean_diff + critical_t * se_diff

    difference_rows.append({
        "source_condition": keys[0],
        "target_condition": keys[1],
        "scenario_type": keys[2],
        "n_seeds": n,
        "mean_detection_rate_difference": mean_diff,
        "sd_difference": sd_diff,
        "ci95_lower": float(ci_low),
        "ci95_upper": float(ci_high),
        "seeds_mahalanobis_higher": int((diffs > 0).sum()),
        "seeds_euclidean_higher": int((diffs < 0).sum()),
        "seeds_tied": int((diffs == 0).sum())
    })

detection_difference_df = pd.DataFrame(difference_rows)

# Summarize control FPR for both detectors by scenario
fpr_pivot = control_seed_summary_df.pivot(
    index=["seed", "source_condition", "target_condition", "scenario_type"],
    columns="detector",
    values="observed_false_alarm_rate"
).reset_index()

fpr_pivot["mahalanobis_minus_euclidean_fpr"] = (
    fpr_pivot["mahalanobis"] - fpr_pivot["euclidean"]
)

fpr_summary_df = (
    fpr_pivot.groupby(group_cols, as_index=False)
    .agg(
        mean_euclidean_fpr=("euclidean", "mean"),
        mean_mahalanobis_fpr=("mahalanobis", "mean"),
        mean_fpr_difference=("mahalanobis_minus_euclidean_fpr", "mean")
    )
)

detection_difference_df.to_csv(
    RESULTS_DIR / "detection_rate_differences.csv", index=False
)
fpr_summary_df.to_csv(
    RESULTS_DIR / "false_alarm_rate_comparison.csv", index=False
)

print("Paired scenario comparisons:", len(detection_difference_df))
print("Expected comparisons:", len(TRANSFER_SCENARIOS))
print("\nDetection-rate differences and 95% confidence intervals:")
display(
    detection_difference_df[
        group_cols + [
            "mean_detection_rate_difference",
            "ci95_lower",
            "ci95_upper",
            "seeds_mahalanobis_higher",
            "seeds_euclidean_higher"
        ]
    ].round(4)
)

print("\nFalse-alarm rate comparison:")
display(
    fpr_summary_df.round(4)
)

assert len(detection_difference_df) == len(TRANSFER_SCENARIOS)
assert len(fpr_summary_df) == len(TRANSFER_SCENARIOS)
assert np.isfinite(
    detection_difference_df[
        ["mean_detection_rate_difference", "ci95_lower", "ci95_upper"]
    ].to_numpy()
).all()
assert (detection_difference_df["n_seeds"] == CONFIG["n_seeds"]).all()

print("Statistical comparison validation passed.")
print("Saved paired detection differences and false-alarm comparison.")


Paired scenario comparisons: 9
Expected comparisons: 9

Detection-rate differences and 95% confidence intervals:


,source_condition,target_condition,scenario_type,mean_detection_rate_difference,ci95_lower,ci95_upper,seeds_mahalanobis_higher,seeds_euclidean_higher
0,anisotropic,anisotropic,same_condition,0.0504,0.0333,0.0675,19,1
1,anisotropic,correlated,cross_condition,0.4444,0.4256,0.4632,20,0
2,anisotropic,isotropic,cross_condition,0.4844,0.4585,0.5103,20,0
3,correlated,anisotropic,cross_condition,0.5808,0.5462,0.6154,20,0
4,correlated,correlated,same_condition,0.1449,0.1211,0.1687,20,0
5,correlated,isotropic,cross_condition,0.9551,0.9477,0.9625,20,0
6,isotropic,anisotropic,cross_condition,-0.0002,-0.0011,0.0007,2,4
7,isotropic,correlated,cross_condition,-0.0001,-0.0007,0.0005,2,2
8,isotropic,isotropic,same_condition,-0.0004,-0.0009,0.0001,0,3



False-alarm rate comparison:


,source_condition,target_condition,scenario_type,mean_euclidean_fpr,mean_mahalanobis_fpr,mean_fpr_difference
0,anisotropic,anisotropic,same_condition,0.0507,0.0516,0.0009
1,anisotropic,correlated,cross_condition,0.0028,0.3538,0.3510
2,anisotropic,isotropic,cross_condition,0.0002,0.3408,0.3406
3,correlated,anisotropic,cross_condition,0.3284,0.9950,0.6666
4,correlated,correlated,same_condition,0.0488,0.0530,0.0042
5,correlated,isotropic,cross_condition,0.0084,0.9667,0.9583
6,isotropic,anisotropic,cross_condition,0.5415,0.5410,-0.0005
7,isotropic,correlated,cross_condition,0.1210,0.1209,-0.0001
8,isotropic,isotropic,same_condition,0.0575,0.0573,-0.0002


Statistical comparison validation passed.
Saved paired detection differences and false-alarm comparison.


## Step 9: Experiment Integrity Audit

This stage verifies the completeness and numerical validity of the Experiment 16 outputs.

The audit checks the expected counts of reference datasets, calibration scores, frozen thresholds, independent control windows, drift evaluation windows, seed-level summaries, and statistical comparisons.

It also checks for non-finite scores and thresholds, missing values in key result files, and consistency between the configured number of seeds and the generated results.

The audit is intended to identify implementation and record-keeping errors. Passing the audit does not independently establish scientific validity or real-world generalisability.

In [11]:

# Experiment 16 — Step 9: Final integrity audit

audit_checks = []

def record_check(name, passed, observed, expected):
    audit_checks.append({
        "check": name,
        "passed": bool(passed),
        "observed": str(observed),
        "expected": str(expected)
    })

expected_seed_condition_pairs = CONFIG["n_seeds"] * len(CONDITIONS)
expected_thresholds = expected_seed_condition_pairs * len(CONFIG["detectors"])
expected_calibration_scores = expected_thresholds * CONFIG["calibration_windows"]
expected_control_windows = (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS)
    * len(CONFIG["detectors"]) * CONFIG["control_windows"]
)
expected_drift_windows = (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS)
    * len(CONFIG["detectors"])
    * CONFIG["evaluation_windows_per_condition"]
)
expected_summaries = (
    CONFIG["n_seeds"] * len(TRANSFER_SCENARIOS) * len(CONFIG["detectors"])
)

record_check(
    "reference_dataset_count",
    len(REFERENCE_DATASETS) == expected_seed_condition_pairs,
    len(REFERENCE_DATASETS),
    expected_seed_condition_pairs
)
record_check(
    "calibration_score_count",
    len(calibration_df) == expected_calibration_scores,
    len(calibration_df),
    expected_calibration_scores
)
record_check(
    "threshold_count",
    len(thresholds_df) == expected_thresholds,
    len(thresholds_df),
    expected_thresholds
)
record_check(
    "control_window_count",
    len(control_results_df) == expected_control_windows,
    len(control_results_df),
    expected_control_windows
)
record_check(
    "control_summary_count",
    len(control_seed_summary_df) == expected_summaries,
    len(control_seed_summary_df),
    expected_summaries
)
record_check(
    "drift_window_count",
    len(drift_results_df) == expected_drift_windows,
    len(drift_results_df),
    expected_drift_windows
)
record_check(
    "drift_summary_count",
    len(drift_seed_summary_df) == expected_summaries,
    len(drift_seed_summary_df),
    expected_summaries
)
record_check(
    "scenario_count",
    len(transfer_scenarios_df) == 9,
    len(transfer_scenarios_df),
    9
)
record_check(
    "statistical_comparison_count",
    len(detection_difference_df) == 9,
    len(detection_difference_df),
    9
)
record_check(
    "all_calibration_scores_finite",
    np.isfinite(calibration_df["score"]).all(),
    np.isfinite(calibration_df["score"]).all(),
    True
)
record_check(
    "all_thresholds_finite",
    np.isfinite(thresholds_df["threshold"]).all(),
    np.isfinite(thresholds_df["threshold"]).all(),
    True
)
record_check(
    "all_control_scores_finite",
    np.isfinite(control_results_df["score"]).all(),
    np.isfinite(control_results_df["score"]).all(),
    True
)
record_check(
    "all_drift_scores_finite",
    np.isfinite(drift_results_df["score"]).all(),
    np.isfinite(drift_results_df["score"]).all(),
    True
)
record_check(
    "control_rates_within_bounds",
    control_seed_summary_df["observed_false_alarm_rate"].between(0, 1).all(),
    control_seed_summary_df["observed_false_alarm_rate"].between(0, 1).all(),
    True
)
record_check(
    "detection_rates_within_bounds",
    drift_seed_summary_df["detection_rate"].between(0, 1).all(),
    drift_seed_summary_df["detection_rate"].between(0, 1).all(),
    True
)

audit_df = pd.DataFrame(audit_checks)
audit_df.to_csv(
    RESULTS_DIR / "experiment_16_integrity_audit.csv",
    index=False
)

display(audit_df)

print("Checks passed:", int(audit_df["passed"].sum()), "/", len(audit_df))
print("Audit file:", RESULTS_DIR / "experiment_16_integrity_audit.csv")

assert audit_df["passed"].all(), (
    "Integrity audit failed. Inspect the failed checks before proceeding."
)

print("Experiment 16 integrity audit passed.")


,check,passed,observed,expected
0,reference_dataset_count,True,60,60
1,calibration_score_count,True,24000,24000
2,threshold_count,True,120,120
3,control_window_count,True,180000,180000
4,control_summary_count,True,360,360
5,drift_window_count,True,180000,180000
6,drift_summary_count,True,360,360
7,scenario_count,True,9,9
8,statistical_comparison_count,True,9,9
9,all_calibration_scores_finite,True,True,True


Checks passed: 15 / 15
Audit file: /content/results/experiment_16/experiment_16_integrity_audit.csv
Experiment 16 integrity audit passed.



# Experiment 16 — Step 10: Target-Specific Recalibration Thresholds

## Objective

This step investigates whether target-specific calibration can reduce false-alarm inflation caused by transferring embedding-level drift detectors across different statistical conditions.

The original experiment uses frozen source-condition calibration thresholds. This step introduces two additional calibration strategies to evaluate whether adapting the threshold or the detector configuration to the target condition improves calibration transfer.

## Recalibration Strategies

Two additional methods are evaluated:

**Method B — Source Detector with Target-Specific Threshold**

The detector parameters, including the reference mean and regularized covariance estimate, remain fitted to the source condition. However, the detection threshold is recalculated using independent calibration windows generated under the target condition.

This method evaluates whether threshold adaptation alone can improve cross-condition calibration.

**Method C — Target Detector with Target-Specific Threshold**

The detector parameters are obtained from the target-condition reference dataset. The detection threshold is then calculated using target-condition calibration windows.

This method provides a target-adapted benchmark against which the frozen source detector and threshold can be compared.

## Calibration Procedure

For each random seed and target condition:

1. Generate target-condition calibration windows using the same deterministic random-number generation procedure established in Step 5.
2. Calculate Euclidean and Mahalanobis detector scores.
3. Estimate the recalibrated threshold using the empirical 95th percentile of the calibration scores.
4. Store the thresholds separately for Methods B and C.
5. Validate the number, uniqueness, and numerical validity of the generated threshold records.

The nominal target false-alarm rate is 5%.

The target-specific calibration windows are separate from the independent control and drift evaluation windows used in subsequent steps.

## Experimental Configuration

- Random seeds: 20
- Source conditions: Isotropic, anisotropic, and correlated
- Target conditions: Isotropic, anisotropic, and correlated
- Detectors: Euclidean and Ledoit-Wolf-regularized Mahalanobis
- Calibration windows per seed and target condition: 200
- Window size: 20
- Nominal false-alarm rate: 5%

The procedure generates 720 threshold records across the two recalibration methods.

## Output

The generated threshold records are saved as:

`target_adaptive_thresholds.csv`

The output contains the random seed, source condition, target condition, detector, calibration method, estimated threshold, and calibration configuration.

## Interpretation and Limitations

This step establishes alternative calibration thresholds; it does not independently demonstrate improved false-alarm control or drift detection performance.

The effectiveness of the recalibration strategies must be assessed using independent control and drift evaluation windows in subsequent steps.

All embeddings are synthetically generated from specified multivariate Gaussian distributions. Therefore, the results characterize statistical behaviour under controlled simulation assumptions and do not establish real-world biometric recognition accuracy, fairness, or deployment readiness.


In [12]:

# Experiment 16 — Step 10: Target-Specific Recalibration Thresholds

TARGET_THRESHOLD_ROWS = []

n_calibration = CONFIG["calibration_windows"]
target_fpr = CONFIG["target_false_alarm_rate"]

# Generate target calibration windows once per seed and target condition.
for seed in range(
    CONFIG["random_seed"],
    CONFIG["random_seed"] + CONFIG["n_seeds"]
):
    for target in CONDITIONS:

        # Fit already-existing source and target detectors.
        target_ref_mean, target_precision = FITTED_DETECTORS[(seed, target)]

        # Reproduce the target-condition calibration windows
        # using the same deterministic RNG procedure as Step 5.
        rng = np.random.default_rng(
            seed * 10000 + CONDITIONS.index(target)
        )

        target_calibration_means = generate_windows(
            mean=np.zeros(D),
            covariance=COVARIANCE_MATRICES[target],
            n_windows=n_calibration,
            window_size=window_size,
            rng=rng
        )

        for source in CONDITIONS:

            source_ref_mean, source_precision = FITTED_DETECTORS[(seed, source)]

            for detector in CONFIG["detectors"]:

                # Method B:
                # Source detector with target-specific calibration threshold.
                source_scores = detector_scores(
                    target_calibration_means,
                    source_ref_mean,
                    source_precision,
                    detector
                )

                source_adaptive_threshold = float(
                    np.quantile(source_scores, 1 - target_fpr)
                )

                TARGET_THRESHOLD_ROWS.append({
                    "seed": seed,
                    "source_condition": source,
                    "target_condition": target,
                    "detector": detector,
                    "method": "B_source_detector_target_threshold",
                    "threshold": source_adaptive_threshold,
                    "calibration_windows": n_calibration,
                    "target_false_alarm_rate": target_fpr
                })

                # Method C:
                # Target-fitted detector with target-specific threshold.
                target_scores = detector_scores(
                    target_calibration_means,
                    target_ref_mean,
                    target_precision,
                    detector
                )

                target_adaptive_threshold = float(
                    np.quantile(target_scores, 1 - target_fpr)
                )

                TARGET_THRESHOLD_ROWS.append({
                    "seed": seed,
                    "source_condition": source,
                    "target_condition": target,
                    "detector": detector,
                    "method": "C_target_detector_target_threshold",
                    "threshold": target_adaptive_threshold,
                    "calibration_windows": n_calibration,
                    "target_false_alarm_rate": target_fpr
                })


# Convert results into a DataFrame.
target_thresholds_df = pd.DataFrame(TARGET_THRESHOLD_ROWS)

# Save without overwriting the original frozen thresholds.
target_thresholds_df.to_csv(
    RESULTS_DIR / "target_adaptive_thresholds.csv",
    index=False
)

# Expected number of threshold records.
expected_thresholds = (
    CONFIG["n_seeds"]
    * len(CONDITIONS)
    * len(CONDITIONS)
    * len(CONFIG["detectors"])
    * 2
)

# Basic validation.
print("Target-adaptive thresholds:", len(target_thresholds_df))
print("Expected thresholds:", expected_thresholds)
print(
    "All thresholds finite:",
    np.isfinite(target_thresholds_df["threshold"]).all()
)

print(
    "Unique seed-source-target-detector-method combinations:",
    target_thresholds_df[
        ["seed", "source_condition", "target_condition",
         "detector", "method"]
    ].drop_duplicates().shape[0]
)

display(
    target_thresholds_df.groupby(
        ["method", "source_condition", "target_condition", "detector"]
    )["threshold"]
    .agg(["mean", "std"])
    .round(4)
)

# Assertions.
assert len(target_thresholds_df) == expected_thresholds

assert np.isfinite(
    target_thresholds_df["threshold"]
).all()

assert not target_thresholds_df.duplicated(
    subset=[
        "seed",
        "source_condition",
        "target_condition",
        "detector",
        "method"
    ]
).any()

assert set(target_thresholds_df["method"]) == {
    "B_source_detector_target_threshold",
    "C_target_detector_target_threshold"
}

print("\nStep 10 validation passed.")
print("Saved target-adaptive thresholds.")


Target-adaptive thresholds: 720
Expected thresholds: 720
All thresholds finite: True
Unique seed-source-target-detector-method combinations: 720


mean  \
method                             source_condition target_condition detector              
B_source_detector_target_threshold anisotropic      anisotropic      euclidean    2.4881   
                                                                     mahalanobis  2.0532   
                                                    correlated       euclidean    2.1914   
                                                                     mahalanobis  2.4758   
                                                    isotropic        euclidean    2.0689   
                                                                     mahalanobis  2.3327   
                                   correlated       anisotropic      euclidean    2.4744   
                                                                     mahalanobis  3.5729   
                                                    correlated       euclidean    2.1902   
                                                                     mahalanobis  2.0688   
                                                    isotropic        euclidean    2.0565   
                                                                     mahalanobis  2.9611   
                                   isotropic        anisotropic      euclidean    2.4760   
                                                                     mahalanobis  2.4774   
                                                    correlated       euclidean    2.1813   
                                                                     mahalanobis  2.1831   
                                                    isotropic        euclidean    2.0565   
                                                                     mahalanobis  2.0580   
C_target_detector_target_threshold anisotropic      anisotropic      euclidean    2.4881   
                                                                     mahalanobis  2.0532   
                                                    correlated       euclidean    2.1902   
                                                                     mahalanobis  2.0688   
                                                    isotropic        euclidean    2.0565   
                                                                     mahalanobis  2.0580   
                                   correlated       anisotropic      euclidean    2.4881   
                                                                     mahalanobis  2.0532   
                                                    correlated       euclidean    2.1902   
                                                                     mahalanobis  2.0688   
                                                    isotropic        euclidean    2.0565   
                                                                     mahalanobis  2.0580   
                                   isotropic        anisotropic      euclidean    2.4881   
                                                                     mahalanobis  2.0532   
                                                    correlated       euclidean    2.1902   
                                                                     mahalanobis  2.0688   
                                                    isotropic        euclidean    2.0565   
                                                                     mahalanobis  2.0580   

                                                                                     std  
method                             source_condition target_condition detector             
B_source_detector_target_threshold anisotropic      anisotropic      euclidean    0.0331  
                                                                     mahalanobis  0.0184  
                                                    correlated       euclidean    0.0438  
                                                                     mahalanobis  0.0589  
                                         


Step 10 validation passed.
Saved target-adaptive thresholds.


In [13]:
print("Target thresholds:", len(target_thresholds_df))
print("Reference datasets:", len(REFERENCE_DATASETS))
print("Fitted detectors:", len(FITTED_DETECTORS))
print("Control results:", len(control_results_df))
print("Drift results:", len(drift_results_df))
print("Covariance conditions:", list(COVARIANCE_MATRICES.keys()))

Target thresholds: 720
Reference datasets: 60
Fitted detectors: 60
Control results: 180000
Drift results: 180000
Covariance conditions: ['isotropic', 'anisotropic', 'correlated']



# Experiment 16 — Step 11: Shared Control and Drift Evaluation Datasets

## Objective

This step creates independent control and drift evaluation datasets for comparing the three calibration strategies introduced in the experiment.

To ensure a controlled comparison, identical evaluation windows are shared across all source conditions, detectors, and calibration methods for each random seed and target condition.

## Dataset Generation

For each random seed and target condition:

1. Generate independent control windows from the target-condition distribution.
2. Generate drift evaluation windows using the target covariance structure and configured mean-shift strength.
3. Generate one reproducible drift direction for each seed and target condition.
4. Reuse the same control and drift windows across all calibration strategies.

## Independence and Reproducibility

The evaluation datasets use random-number generator streams separate from the calibration procedure.

The shared evaluation design prevents differences in random evaluation samples from confounding comparisons between calibration methods.

## Experimental Configuration

- Random seeds: 20
- Target conditions: 3
- Control windows per seed and target: 500
- Drift evaluation windows per seed and target: 500
- Embedding dimension: 64
- Window size: 20
- Mean-shift strength: 0.5

## Output

- `shared_control_windows.npz`
- `shared_drift_windows.npz`
- `shared_evaluation_dataset_manifest.csv`

These datasets will be used in Step 12 to evaluate the frozen-source and target-adaptive calibration strategies.


In [14]:

# Experiment 16 — Step 11: Shared Control and Drift Evaluation Datasets

SHARED_CONTROL_WINDOWS = {}
SHARED_DRIFT_WINDOWS = {}
SHARED_DRIFT_DIRECTIONS = {}

MANIFEST_ROWS = []

n_control = CONFIG["control_windows"]
n_eval = CONFIG["evaluation_windows_per_condition"]
window_size = CONFIG["window_size"]
shift_strength = CONFIG["mean_shift_strength"]

for seed in range(
    CONFIG["random_seed"],
    CONFIG["random_seed"] + CONFIG["n_seeds"]
):
    for target in CONDITIONS:

        target_idx = CONDITIONS.index(target)
        target_cov = COVARIANCE_MATRICES[target]

        # Independent control evaluation windows.
        control_rng = np.random.default_rng(
            seed * 400000 + target_idx + 110000
        )

        control_means = generate_windows(
            mean=np.zeros(D),
            covariance=target_cov,
            n_windows=n_control,
            window_size=window_size,
            rng=control_rng
        )

        # One shared drift direction per seed and target condition.
        direction_rng = np.random.default_rng(
            seed * 500000 + target_idx + 120000
        )

        direction = direction_rng.normal(size=D)
        direction /= np.linalg.norm(direction)

        shift = shift_strength * direction

        # Independent drift evaluation windows.
        drift_rng = np.random.default_rng(
            seed * 600000 + target_idx + 130000
        )

        drift_means = generate_windows(
            mean=shift,
            covariance=target_cov,
            n_windows=n_eval,
            window_size=window_size,
            rng=drift_rng
        )

        # Store shared datasets.
        SHARED_CONTROL_WINDOWS[(seed, target)] = control_means
        SHARED_DRIFT_WINDOWS[(seed, target)] = drift_means
        SHARED_DRIFT_DIRECTIONS[(seed, target)] = direction

        MANIFEST_ROWS.append({
            "seed": seed,
            "target_condition": target,
            "n_control_windows": len(control_means),
            "n_drift_windows": len(drift_means),
            "embedding_dimension": D,
            "window_size": window_size,
            "mean_shift_strength": shift_strength,
            "control_all_finite": bool(np.isfinite(control_means).all()),
            "drift_all_finite": bool(np.isfinite(drift_means).all()),
            "drift_direction_norm": float(np.linalg.norm(direction))
        })


# Create manifest.
shared_manifest_df = pd.DataFrame(MANIFEST_ROWS)

# Save reusable evaluation datasets.
np.savez_compressed(
    RESULTS_DIR / "shared_control_windows.npz",
    **{
        f"seed_{seed}_{target}": values
        for (seed, target), values in SHARED_CONTROL_WINDOWS.items()
    }
)

np.savez_compressed(
    RESULTS_DIR / "shared_drift_windows.npz",
    **{
        f"seed_{seed}_{target}": values
        for (seed, target), values in SHARED_DRIFT_WINDOWS.items()
    }
)

shared_manifest_df.to_csv(
    RESULTS_DIR / "shared_evaluation_dataset_manifest.csv",
    index=False
)

# Validation.
expected_datasets = CONFIG["n_seeds"] * len(CONDITIONS)

print("Shared control datasets:", len(SHARED_CONTROL_WINDOWS))
print("Shared drift datasets:", len(SHARED_DRIFT_WINDOWS))
print("Expected datasets:", expected_datasets)

print(
    "Total control windows:",
    sum(len(x) for x in SHARED_CONTROL_WINDOWS.values())
)

print(
    "Total drift windows:",
    sum(len(x) for x in SHARED_DRIFT_WINDOWS.values())
)

print(
    "All control values finite:",
    all(np.isfinite(x).all() for x in SHARED_CONTROL_WINDOWS.values())
)

print(
    "All drift values finite:",
    all(np.isfinite(x).all() for x in SHARED_DRIFT_WINDOWS.values())
)

print(
    "All drift directions normalized:",
    all(
        np.isclose(np.linalg.norm(x), 1.0)
        for x in SHARED_DRIFT_DIRECTIONS.values()
    )
)

assert len(SHARED_CONTROL_WINDOWS) == expected_datasets
assert len(SHARED_DRIFT_WINDOWS) == expected_datasets
assert len(shared_manifest_df) == expected_datasets

assert all(
    x.shape == (n_control, D)
    for x in SHARED_CONTROL_WINDOWS.values()
)

assert all(
    x.shape == (n_eval, D)
    for x in SHARED_DRIFT_WINDOWS.values()
)

assert all(
    np.isfinite(x).all()
    for x in SHARED_CONTROL_WINDOWS.values()
)

assert all(
    np.isfinite(x).all()
    for x in SHARED_DRIFT_WINDOWS.values()
)

assert all(
    np.isclose(np.linalg.norm(x), 1.0)
    for x in SHARED_DRIFT_DIRECTIONS.values()
)

print("\nStep 11 validation passed.")
print("Shared evaluation datasets saved successfully.")


Shared control datasets: 60
Shared drift datasets: 60
Expected datasets: 60
Total control windows: 30000
Total drift windows: 30000
All control values finite: True
All drift values finite: True
All drift directions normalized: True

Step 11 validation passed.
Shared evaluation datasets saved successfully.



# Experiment 16 — Step 12: Comparative Evaluation of Calibration Strategies

## Objective

This step evaluates the three calibration strategies using the shared independent control and drift evaluation datasets generated in Step 11.

The purpose is to determine whether target-specific threshold recalibration and target-specific detector fitting improve calibration-transfer performance compared with the original frozen-source approach.

## Evaluation Strategies

**Method A — Frozen Source Calibration**

Uses the source-condition reference mean, regularized covariance estimate, and frozen source-condition calibration threshold.

**Method B — Target-Specific Threshold Recalibration**

Retains the source-condition detector parameters but recalculates the threshold using target-condition calibration windows.

**Method C — Full Target Adaptation**

Uses target-condition detector parameters and target-condition calibration thresholds.

## Evaluation Procedure

For each random seed, source condition, target condition, and detector:

1. Retrieve the corresponding detector parameters.
2. Retrieve the threshold associated with each calibration strategy.
3. Score the shared independent control windows.
4. Calculate false alarms using the applicable threshold.
5. Score the shared drift evaluation windows.
6. Calculate detected drift windows using the same threshold.
7. Record seed-level performance metrics for all three strategies.

The same evaluation windows are used across all three methods to support paired comparisons.

## Evaluation Metrics

- False-alarm count and false-alarm rate.
- Detected drift-window count and detection rate.
- Across-seed variability.
- Performance differences between calibration strategies.

## Expected Output

The experiment produces 1,080 seed-level evaluation records, representing:

20 random seeds × 3 source conditions × 3 target conditions × 2 detectors × 3 methods.

The results are saved separately for control evaluation, drift evaluation, and combined performance summaries.

## Interpretation and Limitations

A lower false-alarm rate does not necessarily imply improved drift detection sensitivity. Both metrics must be considered together.

Method C represents a target-adapted benchmark and assumes access to target-condition reference and calibration data. It is not equivalent to zero-shot source-to-target transfer.

All results remain limited to the specified synthetic multivariate Gaussian simulation conditions.


In [16]:
# ============================================================
# EXPERIMENT 16 — STEP 12: COMPARATIVE EVALUATION SUMMARY
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Validate input DataFrames
# ------------------------------------------------------------

required_control_columns = [
    "seed",
    "source_condition",
    "target_condition",
    "detector",
    "method",
    "observed_false_alarm_rate",
    "false_alarms",
    "n_control_windows"
]

required_drift_columns = [
    "seed",
    "source_condition",
    "target_condition",
    "detector",
    "method",
    "detection_rate",
    "detected_windows",
    "n_evaluation_windows"
]

missing_control = [
    col for col in required_control_columns
    if col not in comparative_control_df.columns
]

missing_drift = [
    col for col in required_drift_columns
    if col not in comparative_drift_df.columns
]

if missing_control:
    raise ValueError(
        f"Missing control columns: {missing_control}"
    )

if missing_drift:
    raise ValueError(
        f"Missing drift columns: {missing_drift}"
    )

# ------------------------------------------------------------
# 2. Merge control and drift evaluation results
# ------------------------------------------------------------

merge_keys = [
    "seed",
    "source_condition",
    "target_condition",
    "detector",
    "method"
]

comparative_summary_df = comparative_control_df.merge(
    comparative_drift_df[
        merge_keys
        + [
            "detection_rate",
            "detected_windows",
            "n_evaluation_windows"
        ]
    ],
    on=merge_keys,
    how="inner",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 3. Validate merged evaluation records
# ------------------------------------------------------------

expected_records = (
    CONFIG["n_seeds"]
    * len(CONDITIONS)
    * len(CONDITIONS)
    * len(CONFIG["detectors"])
    * 3
)

if len(comparative_control_df) != expected_records:
    raise ValueError(
        f"Unexpected control records: "
        f"{len(comparative_control_df)}; "
        f"expected {expected_records}"
    )

if len(comparative_drift_df) != expected_records:
    raise ValueError(
        f"Unexpected drift records: "
        f"{len(comparative_drift_df)}; "
        f"expected {expected_records}"
    )

if len(comparative_summary_df) != expected_records:
    raise ValueError(
        f"Unexpected merged records: "
        f"{len(comparative_summary_df)}; "
        f"expected {expected_records}"
    )

# ------------------------------------------------------------
# 4. Aggregate results across random seeds
# ------------------------------------------------------------

group_columns = [
    "source_condition",
    "target_condition",
    "detector",
    "method"
]

comparative_summary_df["scenario_type"] = np.where(
    comparative_summary_df["source_condition"]
    == comparative_summary_df["target_condition"],
    "same_condition",
    "cross_condition"
)

group_columns = [
    "source_condition",
    "target_condition",
    "scenario_type",
    "detector",
    "method"
]

comparative_summary_df = (
    comparative_summary_df
    .groupby(group_columns)
    .agg(
        mean_false_alarm_rate=(
            "observed_false_alarm_rate",
            "mean"
        ),
        sd_false_alarm_rate=(
            "observed_false_alarm_rate",
            "std"
        ),
        mean_detection_rate=(
            "detection_rate",
            "mean"
        ),
        sd_detection_rate=(
            "detection_rate",
            "std"
        ),
        mean_false_alarms=(
            "false_alarms",
            "mean"
        ),
        mean_detected_windows=(
            "detected_windows",
            "mean"
        ),
        total_control_windows=(
            "n_control_windows",
            "sum"
        ),
        total_evaluation_windows=(
            "n_evaluation_windows",
            "sum"
        ),
        n_seed_replicates=(
            "seed",
            "nunique"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. Validate aggregated results
# ------------------------------------------------------------

expected_summary_rows = (
    len(CONDITIONS)
    * len(CONDITIONS)
    * len(CONFIG["detectors"])
    * 3
)

if len(comparative_summary_df) != expected_summary_rows:
    raise ValueError(
        f"Unexpected summary rows: "
        f"{len(comparative_summary_df)}; "
        f"expected {expected_summary_rows}"
    )

rate_columns = [
    "mean_false_alarm_rate",
    "sd_false_alarm_rate",
    "mean_detection_rate",
    "sd_detection_rate"
]

if not np.isfinite(
    comparative_summary_df[rate_columns].to_numpy()
).all():
    raise ValueError(
        "Non-finite values detected in comparative summary."
    )

if not comparative_summary_df[
    "mean_false_alarm_rate"
].between(0, 1).all():
    raise ValueError(
        "False-alarm rates outside the valid [0, 1] range."
    )

if not comparative_summary_df[
    "mean_detection_rate"
].between(0, 1).all():
    raise ValueError(
        "Detection rates outside the valid [0, 1] range."
    )

if not (
    comparative_summary_df["n_seed_replicates"]
    == CONFIG["n_seeds"]
).all():
    raise ValueError(
        "Some comparison groups do not contain all seed replicates."
    )

# ------------------------------------------------------------
# 6. Save comparative evaluation outputs
# ------------------------------------------------------------

comparative_control_df.to_csv(
    RESULTS_DIR / "comparative_control_evaluation.csv",
    index=False
)

comparative_drift_df.to_csv(
    RESULTS_DIR / "comparative_drift_evaluation.csv",
    index=False
)

comparative_summary_df.to_csv(
    RESULTS_DIR / "comparative_calibration_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Print execution summary
# ------------------------------------------------------------

print("=" * 65)
print("EXPERIMENT 16 — STEP 12: COMPARATIVE EVALUATION")
print("=" * 65)

print("Comparative control records:", len(comparative_control_df))
print("Comparative drift records:", len(comparative_drift_df))
print("Expected records:", expected_records)
print("Comparative summary rows:", len(comparative_summary_df))

print("\nMethods evaluated:")
print("Method A: Frozen source detector + frozen source threshold")
print("Method B: Source detector + target-specific threshold")
print("Method C: Target-fitted detector + target-specific threshold")

print("\nMean performance across seeds:")
display(
    comparative_summary_df.sort_values(
        [
            "source_condition",
            "target_condition",
            "detector",
            "method"
        ]
    ).round(4)
)

print("\nSaved files:")
print("1. comparative_control_evaluation.csv")
print("2. comparative_drift_evaluation.csv")
print("3. comparative_calibration_summary.csv")

print("\nValidation:")
print("All comparative records:", len(comparative_summary_df) == expected_records)
print("All summary groups complete:", len(comparative_summary_df) == expected_summary_rows)
print("All rates finite:", np.isfinite(
    comparative_summary_df[rate_columns].to_numpy()
).all())
print("All rates within valid bounds:", (
    comparative_summary_df["mean_false_alarm_rate"].between(0, 1).all()
    and comparative_summary_df["mean_detection_rate"].between(0, 1).all()
))

print("\nSTEP 12 COMPLETED.")
print("=" * 65)

EXPERIMENT 16 — STEP 12: COMPARATIVE EVALUATION
Comparative control records: 1080
Comparative drift records: 1080
Expected records: 1080
Comparative summary rows: 54

Methods evaluated:
Method A: Frozen source detector + frozen source threshold
Method B: Source detector + target-specific threshold
Method C: Target-fitted detector + target-specific threshold

Mean performance across seeds:


,source_condition,target_condition,scenario_type,detector,method,mean_false_alarm_rate,sd_false_alarm_rate,mean_detection_rate,sd_detection_rate,mean_false_alarms,mean_detected_windows,total_control_windows,total_evaluation_windows,n_seed_replicates
0,anisotropic,anisotropic,same_condition,euclidean,A_frozen_source,0.0505,0.0125,0.0838,0.0214,25.25,41.90,10000,10000,20
1,anisotropic,anisotropic,same_condition,euclidean,B_source_detector_target_threshold,0.0505,0.0125,0.0838,0.0214,25.25,41.90,10000,10000,20
2,anisotropic,anisotropic,same_condition,euclidean,C_target_detector_target_threshold,0.0505,0.0125,0.0838,0.0214,25.25,41.90,10000,10000,20
3,anisotropic,anisotropic,same_condition,mahalanobis,A_frozen_source,0.0495,0.0135,0.1317,0.0284,24.75,65.85,10000,10000,20
4,anisotropic,anisotropic,same_condition,mahalanobis,B_source_detector_target_threshold,0.0495,0.0135,0.1317,0.0284,24.75,65.85,10000,10000,20
5,anisotropic,anisotropic,same_condition,mahalanobis,C_target_detector_target_threshold,0.0495,0.0135,0.1317,0.0284,24.75,65.85,10000,10000,20
6,anisotropic,correlated,cross_condition,euclidean,A_frozen_source,0.0026,0.0029,0.0068,0.0048,1.30,3.40,10000,10000,20
7,anisotropic,correlated,cross_condition,euclidean,B_source_detector_target_threshold,0.0529,0.0185,0.0888,0.0336,26.45,44.40,10000,10000,20
8,anisotropic,correlated,cross_condition,euclidean,C_target_detector_target_threshold,0.0515,0.0186,0.0849,0.0242,25.75,42.45,10000,10000,20
9,anisotropic,correlated,cross_condition,mahalanobis,A_frozen_source,0.3552,0.0384,0.4613,0.0433,177.60,230.65,10000,10000,20



Saved files:
1. comparative_control_evaluation.csv
2. comparative_drift_evaluation.csv
3. comparative_calibration_summary.csv

Validation:
All comparative records: False
All summary groups complete: True
All rates finite: True
All rates within valid bounds: True

STEP 12 COMPLETED.


## Step 13: Paired Statistical Comparison of Calibration Methods

This step performs paired statistical comparisons of the three calibration strategies:

- Method A: Frozen source detector and frozen source threshold.
- Method B: Source detector with target-specific threshold recalibration.
- Method C: Target-fitted detector with target-specific threshold.

The analysis uses the 20 independent random-seed replicates and compares methods under identical source-target scenarios and shared evaluation datasets.

The evaluated metrics are:

1. False-alarm rate.
2. Detection rate.

Pairwise comparisons are conducted using the Wilcoxon signed-rank test. Mean paired differences and 95% bootstrap confidence intervals are also calculated. Holm correction is applied to account for multiple comparisons.

These comparisons assess statistical differences within the synthetic experimental setting and do not establish real-world biometric performance.

In [17]:
# ============================================================
# EXPERIMENT 16 — STEP 13:
# PAIRED STATISTICAL COMPARISON OF METHODS A, B AND C
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

METHODS = [
    "A_frozen_source",
    "B_source_detector_target_threshold",
    "C_target_detector_target_threshold"
]

METRICS = [
    "observed_false_alarm_rate",
    "detection_rate"
]

PAIRWISE_COMPARISONS = [
    ("A_frozen_source", "B_source_detector_target_threshold"),
    ("A_frozen_source", "C_target_detector_target_threshold"),
    ("B_source_detector_target_threshold",
     "C_target_detector_target_threshold")
]

N_BOOTSTRAP = 10000
BOOTSTRAP_SEED = 202616

# ------------------------------------------------------------
# 2. Reconstruct seed-level paired evaluation dataset
# ------------------------------------------------------------

merge_keys = [
    "seed",
    "source_condition",
    "target_condition",
    "detector",
    "method"
]

seed_level_df = comparative_control_df.merge(
    comparative_drift_df[
        merge_keys + [
            "detection_rate",
            "detected_windows"
        ]
    ],
    on=merge_keys,
    how="inner",
    validate="one_to_one"
)

expected_records = (
    CONFIG["n_seeds"]
    * len(CONDITIONS)
    * len(CONDITIONS)
    * len(CONFIG["detectors"])
    * len(METHODS)
)

assert len(seed_level_df) == expected_records, (
    f"Expected {expected_records} paired records, "
    f"found {len(seed_level_df)}"
)

assert seed_level_df["seed"].nunique() == CONFIG["n_seeds"]

assert seed_level_df[METRICS].notna().all().all()

# ------------------------------------------------------------
# 3. Bootstrap confidence interval function
# ------------------------------------------------------------

def paired_bootstrap_ci(
    differences,
    n_bootstrap=10000,
    confidence_level=0.95,
    random_seed=202616
):
    differences = np.asarray(differences, dtype=float)

    rng = np.random.default_rng(random_seed)

    n = len(differences)

    bootstrap_means = np.empty(n_bootstrap)

    for i in range(n_bootstrap):
        sampled = rng.choice(
            differences,
            size=n,
            replace=True
        )

        bootstrap_means[i] = np.mean(sampled)

    alpha = 1 - confidence_level

    lower = np.quantile(
        bootstrap_means,
        alpha / 2
    )

    upper = np.quantile(
        bootstrap_means,
        1 - alpha / 2
    )

    return lower, upper


# ------------------------------------------------------------
# 4. Holm multiple-testing correction
# ------------------------------------------------------------

def holm_adjust(p_values):

    p_values = np.asarray(p_values, dtype=float)

    n = len(p_values)

    order = np.argsort(p_values)

    adjusted = np.empty(n)

    running_max = 0.0

    for rank, index in enumerate(order):

        adjusted_value = (
            (n - rank) * p_values[index]
        )

        running_max = max(
            running_max,
            adjusted_value
        )

        adjusted[index] = min(
            running_max,
            1.0
        )

    return adjusted


# ------------------------------------------------------------
# 5. Run paired comparisons
# ------------------------------------------------------------

results = []

scenario_columns = [
    "source_condition",
    "target_condition",
    "detector"
]

rng_seed_offset = 0

for metric in METRICS:

    for source in CONDITIONS:

        for target in CONDITIONS:

            for detector in CONFIG["detectors"]:

                subset = seed_level_df[
                    (seed_level_df["source_condition"] == source)
                    & (seed_level_df["target_condition"] == target)
                    & (seed_level_df["detector"] == detector)
                ]

                pivot = subset.pivot(
                    index="seed",
                    columns="method",
                    values=metric
                )

                pivot = pivot.reindex(columns=METHODS)

                pivot = pivot.dropna()

                if len(pivot) != CONFIG["n_seeds"]:
                    raise ValueError(
                        f"Incomplete paired data: "
                        f"{source} -> {target}, "
                        f"{detector}, {metric}"
                    )

                for method_1, method_2 in PAIRWISE_COMPARISONS:

                    # Positive difference means method_2
                    # has a higher metric than method_1.
                    differences = (
                        pivot[method_2].to_numpy()
                        - pivot[method_1].to_numpy()
                    )

                    mean_difference = np.mean(differences)

                    median_difference = np.median(differences)

                    if np.allclose(differences, 0):

                        statistic = 0.0
                        p_value = 1.0

                    else:

                        test = wilcoxon(
                            pivot[method_1].to_numpy(),
                            pivot[method_2].to_numpy(),
                            alternative="two-sided",
                            zero_method="wilcox",
                            method="auto"
                        )

                        statistic = float(test.statistic)
                        p_value = float(test.pvalue)

                    ci_lower, ci_upper = paired_bootstrap_ci(
                        differences,
                        n_bootstrap=N_BOOTSTRAP,
                        random_seed=(
                            BOOTSTRAP_SEED + rng_seed_offset
                        )
                    )

                    rng_seed_offset += 1

                    results.append({

                        "metric": metric,

                        "source_condition": source,

                        "target_condition": target,

                        "scenario_type": (
                            "same_condition"
                            if source == target
                            else "cross_condition"
                        ),

                        "detector": detector,

                        "method_1": method_1,

                        "method_2": method_2,

                        "n_seed_pairs": len(differences),

                        "mean_method_1": np.mean(
                            pivot[method_1]
                        ),

                        "mean_method_2": np.mean(
                            pivot[method_2]
                        ),

                        "mean_paired_difference": mean_difference,

                        "median_paired_difference": median_difference,

                        "ci_95_lower": ci_lower,

                        "ci_95_upper": ci_upper,

                        "wilcoxon_statistic": statistic,

                        "p_value": p_value
                    })


# ------------------------------------------------------------
# 6. Create statistical results DataFrame
# ------------------------------------------------------------

paired_statistics_df = pd.DataFrame(results)

# ------------------------------------------------------------
# 7. Apply Holm correction separately for each metric
# ------------------------------------------------------------

paired_statistics_df["holm_adjusted_p_value"] = np.nan

for metric in METRICS:

    mask = paired_statistics_df["metric"] == metric

    paired_statistics_df.loc[
        mask,
        "holm_adjusted_p_value"
    ] = holm_adjust(
        paired_statistics_df.loc[
            mask,
            "p_value"
        ].to_numpy()
    )

paired_statistics_df["significant_after_holm"] = (
    paired_statistics_df["holm_adjusted_p_value"] < 0.05
)

# ------------------------------------------------------------
# 8. Validate statistical results
# ------------------------------------------------------------

expected_comparisons = (
    len(METRICS)
    * len(CONDITIONS)
    * len(CONDITIONS)
    * len(CONFIG["detectors"])
    * len(PAIRWISE_COMPARISONS)
)

assert len(paired_statistics_df) == expected_comparisons

assert paired_statistics_df[
    "p_value"
].between(0, 1).all()

assert paired_statistics_df[
    "holm_adjusted_p_value"
].between(0, 1).all()

numeric_columns = [
    "mean_method_1",
    "mean_method_2",
    "mean_paired_difference",
    "median_paired_difference",
    "ci_95_lower",
    "ci_95_upper",
    "wilcoxon_statistic",
    "p_value",
    "holm_adjusted_p_value"
]

assert np.isfinite(
    paired_statistics_df[numeric_columns].to_numpy()
).all()

assert (
    paired_statistics_df["n_seed_pairs"]
    == CONFIG["n_seeds"]
).all()

# ------------------------------------------------------------
# 9. Save statistical results
# ------------------------------------------------------------

paired_statistics_df.to_csv(
    RESULTS_DIR / "paired_statistical_comparisons.csv",
    index=False
)

# ------------------------------------------------------------
# 10. Display results
# ------------------------------------------------------------

print("=" * 70)
print("EXPERIMENT 16 — STEP 13: PAIRED STATISTICAL COMPARISON")
print("=" * 70)

print("Seed-level paired records:", len(seed_level_df))

print("Statistical comparisons:", len(paired_statistics_df))

print("Expected comparisons:", expected_comparisons)

print("Independent seed replicates:", CONFIG["n_seeds"])

print("\nSignificant comparisons after Holm correction:")

print(
    paired_statistics_df[
        "significant_after_holm"
    ].sum()
)

print("\nFalse-alarm rate comparisons:")

display(
    paired_statistics_df[
        paired_statistics_df["metric"]
        == "observed_false_alarm_rate"
    ].round(5)
)

print("\nDetection rate comparisons:")

display(
    paired_statistics_df[
        paired_statistics_df["metric"]
        == "detection_rate"
    ].round(5)
)

print("\nSaved:")
print("paired_statistical_comparisons.csv")

print("\nValidation:")

print(
    "Correct comparison count:",
    len(paired_statistics_df) == expected_comparisons
)

print(
    "All p-values valid:",
    paired_statistics_df["p_value"].between(0, 1).all()
)

print(
    "All adjusted p-values valid:",
    paired_statistics_df[
        "holm_adjusted_p_value"
    ].between(0, 1).all()
)

print(
    "All comparisons use 20 seed pairs:",
    (
        paired_statistics_df["n_seed_pairs"]
        == CONFIG["n_seeds"]
    ).all()
)

print("\nSTEP 13 COMPLETED.")
print("=" * 70)

EXPERIMENT 16 — STEP 13: PAIRED STATISTICAL COMPARISON
Seed-level paired records: 1080
Statistical comparisons: 108
Expected comparisons: 108
Independent seed replicates: 20

Significant comparisons after Holm correction:
51

False-alarm rate comparisons:


,metric,source_condition,target_condition,scenario_type,detector,method_1,method_2,n_seed_pairs,mean_method_1,mean_method_2,mean_paired_difference,median_paired_difference,ci_95_lower,ci_95_upper,wilcoxon_statistic,p_value,holm_adjusted_p_value,significant_after_holm
0,observed_false_alarm_rate,isotropic,isotropic,same_condition,euclidean,A_frozen_source,B_source_detector_target_threshold,20,0.0570,0.0570,0.0000,0.000,0.0000,0.0000,0.0,1.00000,1.00000,False
1,observed_false_alarm_rate,isotropic,isotropic,same_condition,euclidean,A_frozen_source,C_target_detector_target_threshold,20,0.0570,0.0570,0.0000,0.000,0.0000,0.0000,0.0,1.00000,1.00000,False
2,observed_false_alarm_rate,isotropic,isotropic,same_condition,euclidean,B_source_detector_target_threshold,C_target_detector_target_threshold,20,0.0570,0.0570,0.0000,0.000,0.0000,0.0000,0.0,1.00000,1.00000,False
3,observed_false_alarm_rate,isotropic,isotropic,same_condition,mahalanobis,A_frozen_source,B_source_detector_target_threshold,20,0.0570,0.0570,0.0000,0.000,0.0000,0.0000,0.0,1.00000,1.00000,False
4,observed_false_alarm_rate,isotropic,isotropic,same_condition,mahalanobis,A_frozen_source,C_target_detector_target_threshold,20,0.0570,0.0570,0.0000,0.000,0.0000,0.0000,0.0,1.00000,1.00000,False
5,observed_false_alarm_rate,isotropic,isotropic,same_condition,mahalanobis,B_source_detector_target_threshold,C_target_detector_target_threshold,20,0.0570,0.0570,0.0000,0.000,0.0000,0.0000,0.0,1.00000,1.00000,False
6,observed_false_alarm_rate,isotropic,anisotropic,cross_condition,euclidean,A_frozen_source,B_source_detector_target_threshold,20,0.5503,0.0531,-0.4972,-0.499,-0.5162,-0.4782,0.0,0.00009,0.00455,True
7,observed_false_alarm_rate,isotropic,anisotropic,cross_condition,euclidean,A_frozen_source,C_target_detector_target_threshold,20,0.5503,0.0505,-0.4998,-0.503,-0.5186,-0.4793,0.0,0.00009,0.00455,True
8,observed_false_alarm_rate,isotropic,anisotropic,cross_condition,euclidean,B_source_detector_target_threshold,C_target_detector_target_threshold,20,0.0531,0.0505,-0.0026,-0.003,-0.0079,0.0031,49.0,0.32563,1.00000,False
9,observed_false_alarm_rate,isotropic,anisotropic,cross_condition,mahalanobis,A_frozen_source,B_source_detector_target_threshold,20,0.5500,0.0531,-0.4969,-0.498,-0.5163,-0.4773,0.0,0.00009,0.00455,True



Detection rate comparisons:


,metric,source_condition,target_condition,scenario_type,detector,method_1,method_2,n_seed_pairs,mean_method_1,mean_method_2,mean_paired_difference,median_paired_difference,ci_95_lower,ci_95_upper,wilcoxon_statistic,p_value,holm_adjusted_p_value,significant_after_holm
54,detection_rate,isotropic,isotropic,same_condition,euclidean,A_frozen_source,B_source_detector_target_threshold,20,0.1224,0.1224,0.0000,0.000,0.0000,0.0000,0.0,1.00000,1.00000,False
55,detection_rate,isotropic,isotropic,same_condition,euclidean,A_frozen_source,C_target_detector_target_threshold,20,0.1224,0.1224,0.0000,0.000,0.0000,0.0000,0.0,1.00000,1.00000,False
56,detection_rate,isotropic,isotropic,same_condition,euclidean,B_source_detector_target_threshold,C_target_detector_target_threshold,20,0.1224,0.1224,0.0000,0.000,0.0000,0.0000,0.0,1.00000,1.00000,False
57,detection_rate,isotropic,isotropic,same_condition,mahalanobis,A_frozen_source,B_source_detector_target_threshold,20,0.1227,0.1227,0.0000,0.000,0.0000,0.0000,0.0,1.00000,1.00000,False
58,detection_rate,isotropic,isotropic,same_condition,mahalanobis,A_frozen_source,C_target_detector_target_threshold,20,0.1227,0.1227,0.0000,0.000,0.0000,0.0000,0.0,1.00000,1.00000,False
59,detection_rate,isotropic,isotropic,same_condition,mahalanobis,B_source_detector_target_threshold,C_target_detector_target_threshold,20,0.1227,0.1227,0.0000,0.000,0.0000,0.0000,0.0,1.00000,1.00000,False
60,detection_rate,isotropic,anisotropic,cross_condition,euclidean,A_frozen_source,B_source_detector_target_threshold,20,0.6382,0.0858,-0.5524,-0.560,-0.5706,-0.5341,0.0,0.00000,0.00010,True
61,detection_rate,isotropic,anisotropic,cross_condition,euclidean,A_frozen_source,C_target_detector_target_threshold,20,0.6382,0.0838,-0.5544,-0.554,-0.5731,-0.5353,0.0,0.00009,0.00401,True
62,detection_rate,isotropic,anisotropic,cross_condition,euclidean,B_source_detector_target_threshold,C_target_detector_target_threshold,20,0.0858,0.0838,-0.0020,0.001,-0.0111,0.0063,90.5,0.85620,1.00000,False
63,detection_rate,isotropic,anisotropic,cross_condition,mahalanobis,A_frozen_source,B_source_detector_target_threshold,20,0.6378,0.0857,-0.5521,-0.559,-0.5701,-0.5328,0.0,0.00000,0.00010,True



Saved:
paired_statistical_comparisons.csv

Validation:
Correct comparison count: True
All p-values valid: True
All adjusted p-values valid: True
All comparisons use 20 seed pairs: True

STEP 13 COMPLETED.


## Step 14: Results Summary and Limitations

### Experiment Overview

Experiment 16 evaluates cross-condition calibration transfer for Euclidean and Ledoit–Wolf-regularized Mahalanobis embedding-level drift detectors using synthetic multivariate Gaussian data.

The experiment uses:

- 20 independent random seeds.
- 64-dimensional embeddings.
- A window size of 20.
- Three covariance conditions: isotropic, anisotropic, and correlated.
- Nine source-target condition combinations.
- Two drift detectors: Euclidean and Mahalanobis.

The experiment investigates whether calibration thresholds obtained under one embedding distribution remain reliable when applied to another distribution.

### Calibration Strategies

Three calibration strategies are compared:

1. **Method A — Frozen Source Calibration:** Uses the source-condition detector and its frozen source-condition threshold.
2. **Method B — Target-Specific Threshold Recalibration:** Retains the source-condition detector but recalibrates its threshold using target-condition calibration data.
3. **Method C — Target-Fitted Detector and Threshold:** Fits the detector using target-condition reference data and calibrates its threshold using target-condition calibration data.

Method B and Method C require target-condition calibration data. Method C additionally requires target-condition reference data. Therefore, neither should be interpreted as zero-shot transfer.

### Experimental Outputs

The experiment generates:

- Covariance matrix and reference dataset diagnostics.
- Calibration scores and frozen source-condition thresholds.
- Independent control-window scores and false-alarm summaries.
- Drift evaluation scores and detection-rate summaries.
- Target-specific recalibration thresholds.
- Shared control and drift evaluation datasets.
- Comparative control and drift evaluation results for all three methods.
- Comparative calibration performance summaries.
- Paired statistical comparisons using Wilcoxon signed-rank tests.
- Bootstrap confidence intervals for paired mean differences.
- Holm-adjusted p-values for multiple comparisons.
- Integrity and numerical validation outputs.

### Preliminary Findings

**1. Cross-condition calibration transfer**

Frozen source-condition calibration can produce substantial false-alarm inflation or suppression when the target distribution differs from the source distribution.

For example, in the correlated-to-anisotropic transfer scenario using the Mahalanobis detector, Method A produced a mean false-alarm rate of approximately 99.59%, compared with 5.91% for Method B and 4.95% for Method C.

**2. Target-specific threshold recalibration**

Method B substantially improves false-alarm control in the evaluated cross-condition scenarios by recalibrating the threshold while retaining the source detector.

This suggests that threshold recalibration can mitigate some distribution-dependent calibration failures without requiring the detector itself to be refitted.

**3. Target-fitted detector comparison**

Method C also generally brings false-alarm rates closer to the nominal 5% target. However, it does not consistently outperform Method B.

Differences in detection rates between Methods B and C depend on the source-target condition and detector.

**4. Detection performance and false-alarm trade-offs**

Higher detection rates must be interpreted alongside false-alarm rates. A detector that flags nearly all control windows may also detect nearly all drift windows, but this does not demonstrate useful drift discrimination.

**5. Statistical comparisons**

Step 13 produced 108 paired statistical comparisons across the two evaluation metrics, three method pairs, nine source-target scenarios, and two detectors.

The comparisons used 20 paired random-seed replicates, Wilcoxon signed-rank tests, and bootstrap 95% confidence intervals for paired mean differences. Holm correction was applied separately to the false-alarm-rate and detection-rate comparison families.

A total of 51 comparisons were statistically significant after the specified Holm corrections. Statistical significance does not by itself establish practical importance or generalisability.

### Limitations

1. **Synthetic data:** The experiment uses controlled multivariate Gaussian embedding distributions rather than real biometric embeddings.

2. **Controlled drift:** Drift is introduced through synthetic mean shifts. Other forms of distribution shift, including nonlinear, temporal, multimodal, and identity-dependent changes, are not evaluated.

3. **Limited distribution conditions:** Only isotropic, anisotropic, and correlated covariance structures are investigated.

4. **Target-data requirements:** Methods B and C require target-condition calibration data. Method C also requires target-condition reference data. Their performance should not be presented as zero-shot transfer.

5. **Statistical sample size:** Statistical comparisons use 20 independent random-seed replicates. Larger-scale experiments are needed to assess robustness.

6. **Multiple comparisons:** Holm correction is applied separately to the two metric families. The resulting significance claims are conditional on this correction strategy.

7. **No real-world validation:** The experiment does not establish biometric recognition accuracy, demographic fairness, operational robustness, or deployment readiness.

8. **Detector scope:** Only Euclidean and Ledoit–Wolf-regularized Mahalanobis detectors are evaluated.

### Integrity and Reproducibility

The original experiment's 15 programmed integrity checks passed. The subsequent comparative evaluation generated 1,080 control records, 1,080 drift records, and 54 aggregated comparison groups.

The paired statistical analysis generated 108 comparisons, each using 20 seed pairs. Numerical and result-count validations passed.

The experiment outputs are saved as CSV and NPZ files in the Experiment 16 results directory.

These checks establish that the programmed outputs satisfy the tested numerical and structural conditions. They do not independently establish scientific validity, external validity, or real-world generalisability.

### Overall Conclusion

Under the evaluated synthetic conditions, frozen source-condition calibration can be highly sensitive to distribution changes. Target-specific threshold recalibration substantially improves false-alarm control, while target-fitted detector calibration provides an additional adaptation strategy.

The results support further investigation of calibration transfer and target-aware recalibration for embedding-level drift detection. Validation using real biometric embeddings and more diverse distribution-shift scenarios remains necessary.

In [18]:
# ============================================================
# EXPERIMENT 16 — STEP 14: PACKAGE FINAL RESULTS
# ============================================================

from pathlib import Path
import zipfile
from google.colab import files

RESULTS_DIR = Path("/content/results/experiment_16")
ZIP_PATH = Path("/content/experiment_16_results.zip")

# ------------------------------------------------------------
# 1. Validate results directory
# ------------------------------------------------------------

if not RESULTS_DIR.exists():
    raise FileNotFoundError(
        f"Results folder not found: {RESULTS_DIR}"
    )

result_files = sorted(
    path
    for path in RESULTS_DIR.rglob("*")
    if path.is_file()
)

if not result_files:
    raise RuntimeError(
        "No result files found to package."
    )

# ------------------------------------------------------------
# 2. Package all experiment outputs
# ------------------------------------------------------------

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for path in result_files:

        arcname = (
            Path("experiment_16")
            / path.relative_to(RESULTS_DIR)
        )

        zip_file.write(
            path,
            arcname=str(arcname)
        )

# ------------------------------------------------------------
# 3. Validate ZIP archive
# ------------------------------------------------------------

with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:

    packaged_files = zip_file.namelist()

    bad_file = zip_file.testzip()

    if bad_file is not None:
        raise RuntimeError(
            f"ZIP integrity check failed: {bad_file}"
        )

if len(packaged_files) != len(result_files):
    raise RuntimeError(
        "Packaged file count does not match source file count."
    )

# ------------------------------------------------------------
# 4. Print packaging summary
# ------------------------------------------------------------

print("=" * 65)
print("EXPERIMENT 16 — FINAL RESULTS PACKAGE")
print("=" * 65)

print("Source files:", len(result_files))
print("Packaged files:", len(packaged_files))

print(
    "ZIP size (MB):",
    round(ZIP_PATH.stat().st_size / (1024 ** 2), 2)
)

print("ZIP integrity: PASSED")
print("ZIP created:", ZIP_PATH)

print("\nEXPERIMENT 16 PACKAGING COMPLETED.")

# ------------------------------------------------------------
# 5. Download archive
# ------------------------------------------------------------

files.download(str(ZIP_PATH))

EXPERIMENT 16 — FINAL RESULTS PACKAGE
Source files: 24
Packaged files: 24
ZIP size (MB): 32.96
ZIP integrity: PASSED
ZIP created: /content/experiment_16_results.zip

EXPERIMENT 16 PACKAGING COMPLETED.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>